In [ ]:
!pip -q install xgboost
!pip -q install nltk
!pip -q install wordcloud

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 14.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("WELFake_Dataset.csv")

In [ ]:
print(df.shape)

(72134, 4)


In [ ]:
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO],No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others o...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO],"Now, most of the demonstrators gathered last night were exercising their constitutional and protected right to peaceful protest in order to raise issues and create change. Loretta Lynch aka Er...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid",A dozen politically active pastors came here for a private dinner Friday night to hear a conversion story unique in the context of presidential politics: how Louisiana Gov. Bobby Jindal traveled f...,0
4,4,SATAN 2: Russia unvelis an image of its terrifying new ‘SUPERNUKE’ – Western world takes notice,"The RS-28 Sarmat missile, dubbed Satan 2, will replace the SS-18 Flies at 4.3 miles (7km) per sec and with a range of 6,213 miles (10,000km) The weapons are perceived as part of an increasingly ag...",1


In [ ]:
print(df.isnull().sum())

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64


In [ ]:
df = df.dropna()

In [ ]:
print(df.shape)

(71537, 4)


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
print(df["label"].value_counts())

label
1    36509
0    35028
Name: count, dtype: int64


In [ ]:
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")

In [ ]:
df["content"].apply(len)

,content
0,5180
2,354
3,8116
4,2012
5,1609
...,...
72129,4854
72130,3714
72131,2922
72132,3442


In [ ]:
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings("ignore")

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    return " ".join(words)

In [ ]:
df["clean_text"] = df["content"].apply(clean_text)

In [ ]:
df.shape

(71537, 6)

In [ ]:
df = df[df["clean_text"].str.strip() != ""]

In [ ]:
df.shape

(71528, 6)

In [ ]:
print(df.isnull().sum())

Unnamed: 0    0
title         0
text          0
label         0
content       0
clean_text    0
dtype: int64


In [ ]:
df.head()

,Unnamed: 0,title,text,label,content,clean_text
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO],No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others o...,1,LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 o...,law enforcement high alert following threat cop white blacklivesmatter fyf terrorist video comment expected barack obama member fyf fukyoflag blacklivesmatter movement called lynching hanging whit...
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO],"Now, most of the demonstrators gathered last night were exercising their constitutional and protected right to peaceful protest in order to raise issues and create change. Loretta Lynch aka Er...",1,"UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO] Now, most of the demonstrators gathered last night were e...",unbelievable obama’s attorney general say charlotte rioter “peaceful” protesters…in home state north carolina video demonstrator gathered last night exercising constitutional protected right peace...
3,3,"Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid",A dozen politically active pastors came here for a private dinner Friday night to hear a conversion story unique in the context of presidential politics: how Louisiana Gov. Bobby Jindal traveled f...,0,"Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid A dozen politically active pastors came here for a private dinner Friday night to hear a c...",bobby jindal raised hindu us story christian conversion woo evangelicals potential bid dozen politically active pastor came private dinner friday night hear conversion story unique context preside...
4,4,SATAN 2: Russia unvelis an image of its terrifying new ‘SUPERNUKE’ – Western world takes notice,"The RS-28 Sarmat missile, dubbed Satan 2, will replace the SS-18 Flies at 4.3 miles (7km) per sec and with a range of 6,213 miles (10,000km) The weapons are perceived as part of an increasingly ag...",1,"SATAN 2: Russia unvelis an image of its terrifying new ‘SUPERNUKE’ – Western world takes notice The RS-28 Sarmat missile, dubbed Satan 2, will replace the SS-18 Flies at 4.3 miles (7km) per sec an...",satan russia unvelis image terrifying new ‘supernuke’ – western world take notice r sarmat missile dubbed satan replace s fly mile km per sec range mile km weapon perceived part increasingly aggre...
5,5,About Time! Christian Group Sues Amazon and SPLC for Designation as Hate Group,"All we can say on this one is it s about time someone sued the Southern Poverty Law Center!On Tuesday, D. James Kennedy Ministries (DJKM) filed a lawsuit against the Southern Poverty Law Center (S...",1,"About Time! Christian Group Sues Amazon and SPLC for Designation as Hate Group All we can say on this one is it s about time someone sued the Southern Poverty Law Center!On Tuesday, D. James Kenne...",time christian group sue amazon splc designation hate group say one time someone sued southern poverty law centeron tuesday james kennedy ministry djkm filed lawsuit southern poverty law center sp...


In [ ]:
X = df["clean_text"]
y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [ ]:
len(X_train)

57222

In [ ]:
len(X_test)

14306

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)
print(X_train_tfidf.shape)

(57222, 50000)


In [ ]:
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
print(X_test_tfidf.shape)

(14306, 50000)


In [ ]:
len(tfidf.vocabulary_)

50000

In [ ]:
vocab = list(tfidf.vocabulary_.keys())
print(vocab[:100])

['biden', 'trump', 'agree', 'fight', 'pistol', 'arrangement', 'pending', 'thursday', 'october', 'duel', 'seeking', 'duplicate', 'surpass', 'famous', 'vice', 'president', 'aaron', 'burr', 'treasury', 'secretary', 'alexander', 'hamilton', 'republican', 'candidate', 'donald', 'joe', 'agreed', 'although', 'detail', 'yet', 'finalized', 'report', 'likely', 'take', 'place', 'eve', 'election', 'three', 'independent', 'source', 'confirmed', 'negotiation', 'broadcast', 'right', 'extremely', 'tense', 'demand', 'inaugural', 'show', 'new', 'venture', 'tv', 'idea', 'one', 'said', 'could', 'shot', 'someone', 'fifth', 'avenue', 'supporter', 'would', 'hilary', 'obama', 'rubio', 'cruz', 'jeb', 'bush', 'ton', 'others', 'think', 'better', 'hair', 'plug', 'im', 'greatest', 'shooter', 'ever', 'real', 'sniper', 'insist', 'msnbc', 'must', 'broadcaster', 'liberal', 'minority', 'audience', 'want', 'see', 'several', 'gun', 'response', 'way', 'miss', 'glow', 'bright', 'orange', 'point', 'toward', 'megan']


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)


In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    tree_method='hist'
)

In [ ]:
xgb_model.fit(
    X_train_tfidf,
    y_train
)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = xgb_model.predict(X_test_tfidf)
y_prob = xgb_model.predict_proba(X_test_tfidf)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print(f"Accuracy : {accuracy:.4f}")

Accuracy : 0.9715


In [ ]:
precision_score(y_test, y_pred)

0.9623021196672927

In [ ]:
recall_score(y_test, y_pred)

0.9826027397260274

In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97      7006
           1       0.96      0.98      0.97      7300

    accuracy                           0.97     14306
   macro avg       0.97      0.97      0.97     14306
weighted avg       0.97      0.97      0.97     14306



In [ ]:
confusion_matrix(y_test, y_pred)

array([[6725,  281],
       [ 127, 7173]])

In [ ]:
import joblib

In [ ]:
joblib.dump(
    xgb_model,
    "xgboost_model.pkl"
)

['xgboost_model.pkl']

In [ ]:
joblib.dump(
    tfidf,
    "tfidf_vectorizer.pkl"
)

['tfidf_vectorizer.pkl']

In [ ]:
import os
print(os.listdir())

['.config', '.ipynb_checkpoints', 'xgboost_model.pkl', 'WELFake_Dataset.csv', 'tfidf_vectorizer.pkl']


In [ ]:
from google.colab import files
files.download("xgboost_model.pkl")
files.download("tfidf_vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>